# 세션별 기록과 실제 후속 질문

대화 ID별로 기록을 나누고, 같은 대화에서 앞의 내용을 참고해 후속 질문에 답하는 챗봇을 만들어 봅니다. 마지막에는 프로필 정보만 지웠을 때 대화 원문에는 무엇이 남는지 확인합니다.


### 준비. 환경과 저장 도구

- **역할 구분**: `ChatOpenAI`는 모델 연결, `create_agent`는 실행기 구성, `InMemorySaver`는 상태 저장.


In [30]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

## 1. 대화 ID별로 기록 나누기

### 1-1. A/B 대화 실행기 준비·초기화

- **체크포인터(checkpointer)**: 실행 상태를 저장하고, 다음 실행에서 저장된 상태를 불러오는 구성요소.
- **`InMemorySaver()`**: 실행 중인 Python 프로그램의 메모리에 상태를 보관하는 체크포인터.
- **`thread_id`**: 같은 저장소 안에서 대화를 구분하는 식별자. 같은 ID는 기록을 이어 쓰고, 다른 ID는 별도 기록을 사용한다.
- **`create_agent(...)`**: 모델·도구·체크포인터 등을 연결해 Agent를 만드는 함수.
- **저장 범위**: `InMemorySaver`의 기록은 해당 저장소 객체에 보관된다. 새 저장소를 만들면 같은 ID라도 이전 기록을 이어받지 않는다.


In [31]:
model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_completion_tokens=300,
    max_retries=0
)
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="친절한 한국어 도우미",
    checkpointer=InMemorySaver()
)
config_a = {"configurable": {"thread_id": "student-a"}}
config_b = {"configurable": {"thread_id": "student-b"}}

### 1-2. A 자기소개

- **입력**: `messages`에 현재 사용자 메시지 하나, `config`에 대화 A의 설정 전달.
- **상태 연결**: 체크포인터가 같은 ID의 앞 기록을 불러오고 이번 메시지·모델 답변을 반영.
- **첫 요청**: A의 앞 기록이 없으므로 자기소개부터 시작.


In [32]:
result_a_intro = agent.invoke({"messages": [{"role": "user", "content" : '내 이름은 은지야. 인사해줘'}]}, config=config_a)

### 1-3. A 자기소개의 결과

- **반환 타입**: `dict` — 이번 실행이 끝난 상태.
- **`result_a_intro["messages"]`**: 누적 메시지 객체의 리스트.
- **`[-1].content`**: 마지막 메시지 객체의 본문. 처음 한 번 실행했다면 사용자 말 + AI 답변 = 2개.


In [33]:
print(type(result_a_intro))
print(result_a_intro["messages"][-1].content)
print("누적 메시지", len(result_a_intro["messages"]))
print(result_a_intro["messages"])

<class 'dict'>
안녕하세요, 은지님! 만나서 반가워요. 오늘 기분은 어떠세요?
누적 메시지 2
[HumanMessage(content='내 이름은 은지야. 인사해줘', additional_kwargs={}, response_metadata={}, id='b36bfd6c-5443-4cbb-9ae4-517baacb948d'), AIMessage(content='안녕하세요, 은지님! 만나서 반가워요. 오늘 기분은 어떠세요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 30, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1fffd65103', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a21b61-d026-7138-afa2-c4c00d5be35d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 22, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]


### 1-4. B 자기소개와 결과 확인

- **같은 Agent, 다른 ID**: 실행기는 재사용하고 `config_b`로 B의 기록 선택.
- **대화 분리**: A의 ‘민수’와 별개로 B는 ‘수진’부터 시작.
- **반환 구조**: 상태 딕셔너리 → `messages` 리스트 → 마지막 답변 본문.
- **B의 첫 기록**: 자기소개 + AI 답변 = 2개. A의 기록을 합친 개수가 아님.


In [34]:
result_b_intro = agent.invoke(
    {"messages": [
        {"role": "user", "content": "내 이름은 민수야. 인사해줘"}
    ]}, config=config_b
)
print(result_b_intro["messages"])

[HumanMessage(content='내 이름은 민수야. 인사해줘', additional_kwargs={}, response_metadata={}, id='0655b2c0-9704-4a10-b9bd-4b5b0cf585ad'), AIMessage(content='안녕하세요, 민수님! 만나서 반가워요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 30, 'total_tokens': 51, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_8bba7b2773', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a21b61-d573-7426-bb60-eee790a028a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 21, 'total_tokens': 51, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]


### 1-5. A로 돌아와 이름 묻기

- **같은 ID 재사용**: `config_a` → 앞서 A에 저장한 자기소개·답변을 이어 사용.
- **현재 입력**: ‘내 이름이 뭐야?’ 하나만 전달. 이름과 전체 이력을 다시 직접 넣을 필요 없음.
- **답변 근거**: A의 자기소개에 있는 이름 ‘민수’.
- **누적 개수**: 소개·답변 2개 + 새 질문·답변 2개 = 4개.


In [35]:
result_a_name = agent.invoke({"messages": [{"role": "user", "content": '내 이름이 뭐야?'}]}, config=config_a)
print(result_a_name["messages"])

[HumanMessage(content='내 이름은 은지야. 인사해줘', additional_kwargs={}, response_metadata={}, id='b36bfd6c-5443-4cbb-9ae4-517baacb948d'), AIMessage(content='안녕하세요, 은지님! 만나서 반가워요. 오늘 기분은 어떠세요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 30, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1fffd65103', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a21b61-d026-7138-afa2-c4c00d5be35d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 22, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_meta

### 1-6. B의 이름 묻기

- **앞과 같은 처리**: 현재 이름 질문 하나를 보내고 저장된 기록을 이어 사용.
- **달라진 값**: `config_b` → B의 ‘수진’ 자기소개를 참조. 가장 최근 실행한 A의 기록을 고르는 것이 아님.
- **결과 비교**: A는 민수·4개, B는 수진·4개. 답변과 실제 저장 기록을 함께 확인.


In [36]:
result_b_name = agent.invoke({"messages": [{"role": "user", "content": '내 이름이 뭐야?'}]}, config=config_b)
print(result_b_name["messages"][-1].content)

민수님이세요! 기억하고 있어요. 무엇을 도와드릴까요?


### 1-7. 두 ID의 저장 기록 조회

- **`get_state(config)`**: 해당 대화 ID의 현재 상태 스냅샷 조회. 모델을 다시 호출하지 않음.
- **`.values`**: 스냅샷의 상태 딕셔너리. 딕셔너리의 `values()` 메소드와 구분.
- **반복 대상**: 바깥은 ID 문자열 두 개, 안쪽은 해당 ID의 메시지 객체 리스트.
- **읽는 값**: `message.type`은 `human`·`ai` 같은 종류, `message.content`는 본문.
- **세션 분리의 범위**: ID별 기록 구분. 실제 서비스의 사용자 인증·접근 권한 검사는 별도.


In [37]:
for session_id in ("student-a", "student-b"):
    config = {"configurable": {"thread_id": session_id}}
    
    messages = agent.get_state(config).values["messages"]
    
    print(session_id)
    for message in messages:
        print(message.type, message.content)

student-a
human 내 이름은 은지야. 인사해줘
ai 안녕하세요, 은지님! 만나서 반가워요. 오늘 기분은 어떠세요?
human 내 이름이 뭐야?
ai 은지님이세요! 어떻게 도와드릴까요?
student-b
human 내 이름은 민수야. 인사해줘
ai 안녕하세요, 민수님! 만나서 반가워요. 어떻게 도와드릴까요?
human 내 이름이 뭐야?
ai 민수님이세요! 기억하고 있어요. 무엇을 도와드릴까요?


## 2. 앞 대화를 참고해 후속 질문하기

### 2-1. 후속 질문용 별도 대화 준비·초기화

- **후속 질문의 문맥**: 자기소개 → Loader 설명 → Splitter 설명 → ‘그 둘의 차이’.
- **새 저장소**: 모델은 재사용하되 `followup_agent`에는 새 `InMemorySaver()` 연결. A/B 기록을 가져오지 않음.
- **같은 대화 유지**: 네 질문 모두 `followup-demo`라는 하나의 ID 사용.


In [38]:
QUESTIONS = [
    "내 이름은 민수야. Python 초보자이고 RAG를 배우고 있어.",
    "Document Loader의 역할을 두 문장으로 설명해줘",
    "Text Splitter의 역할도 두 문장으로 설명해줘",
    "그 둘의 차이를 Python 초보자가 이해할 수 있는 예시로 다시 설명해줘."
]
followup_agent = create_agent(model=model, tools=[],
                              system_prompt="친절한 AI 수업 도우미입니다. 이전 설명을 참고해 짧고 쉽게 답하세요.",
                              checkpointer=InMemorySaver())
followup_config = {"configurable": {"thread_id": "followup-demo"}}

### 2-2. 첫 소개와 응답 확인

- **`QUESTIONS[0]`**: 첫 자기소개 문자열 — 이름, Python 학습 수준, 배우는 주제.
- **반환 상태**: `messages`의 마지막 객체에서 답변 본문을 읽고 누적 개수 확인.
- **첫 기록**: 사용자 자기소개 + 모델 답변 = 2개. 별도 프로필 필드가 아니라 메시지로 남긴 정보.


In [39]:
followup_first = followup_agent.invoke(
    {"messages": [{"role": "user", "content": QUESTIONS[0]}]},
    config=followup_config
)
print(followup_first["messages"][-1].content)
print("누적 메시지", len(followup_first["messages"]))

안녕 민수야! Python과 RAG 배우는 거 멋지다. 궁금한 점 있으면 언제든 물어봐!
누적 메시지 2


### 2-3. 두 도구의 설명 이어 받기

- **`QUESTIONS[1:3]`**: Loader·Splitter 질문 두 문자열을 담은 리스트. 인덱스 3의 마지막 질문은 제외.
- **반복 처리**: `question`에 문자열 하나씩 → 같은 ID로 호출 → 반환 상태의 마지막 답변 출력.
- **누적 변화**: 앞 2개 → Loader 질문·답변 후 4개 → Splitter 질문·답변 후 6개.
- **답변의 출처**: 미리 적어 둔 예시가 아니라 이번 모델 호출에서 받은 실제 설명.


In [40]:
for question in QUESTIONS[1:3]:
    result = followup_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=followup_config
    )
    print(question)
    print(result["messages"][-1].content)
    print("누적 메시지", len(result["messages"]))


Document Loader의 역할을 두 문장으로 설명해줘
Document Loader는 다양한 형식의 문서를 불러와서 텍스트로 변환하는 역할을 해. 이렇게 변환된 텍스트는 RAG 모델이 이해하고 처리할 수 있도록 준비돼.
누적 메시지 4
Text Splitter의 역할도 두 문장으로 설명해줘
Text Splitter는 긴 텍스트를 작은 단위로 나누는 역할을 해. 이렇게 나눈 텍스트는 모델이 더 쉽게 이해하고 처리할 수 있어.
누적 메시지 6


### 2-4. 마지막 질문에 앞서 기록 확인

- **현재 저장 상태**: 자기소개, Loader, Splitter의 질문·답변 세 쌍 = 6개.
- **반복 대상**: `before_followup`의 메시지 객체를 하나씩 꺼내 종류·본문 출력.
- **아직 보내지 않은 질문**: `QUESTIONS[-1]`을 `last_question`에 저장. 변수에 담거나 출력하는 것은 모델 호출이 아님.


In [41]:
before_followup = followup_agent.get_state(followup_config).values["messages"]
for message in before_followup:
    print(message.type, message.content)

last_question = QUESTIONS[-1]
print("이어서 보낼 질문:", last_question)

human 내 이름은 민수야. Python 초보자이고 RAG를 배우고 있어.
ai 안녕 민수야! Python과 RAG 배우는 거 멋지다. 궁금한 점 있으면 언제든 물어봐!
human Document Loader의 역할을 두 문장으로 설명해줘
ai Document Loader는 다양한 형식의 문서를 불러와서 텍스트로 변환하는 역할을 해. 이렇게 변환된 텍스트는 RAG 모델이 이해하고 처리할 수 있도록 준비돼.
human Text Splitter의 역할도 두 문장으로 설명해줘
ai Text Splitter는 긴 텍스트를 작은 단위로 나누는 역할을 해. 이렇게 나눈 텍스트는 모델이 더 쉽게 이해하고 처리할 수 있어.
이어서 보낼 질문: 그 둘의 차이를 Python 초보자가 이해할 수 있는 예시로 다시 설명해줘.


### 2-5. ‘그 둘’의 차이와 저장 기록 확인

- **문맥 연결**: 같은 ID의 앞 기록을 참고해 ‘그 둘’이 가리키는 Loader·Splitter를 비교.
- **저장 결과**: 사용자 4개 + AI 답변 4개 = 8개. 앞 기록을 지우지 않고 새 차례를 누적.
- **동등 비교**: `saved[0].content == QUESTIONS[0]`은 첫 자기소개 본문이 같은지 확인하는 참·거짓 값.
- **답변과 기록**: 앞의 두 설명과 비교 답변의 의미가 이어지는지, 첫 자기소개도 남아 있는지 대조.


In [42]:
followup_result = followup_agent.invoke(
    {"messages": [{"role": "user", "content": last_question}]},
    config=followup_config
)
print(followup_result["messages"][-1].content)
saved = followup_agent.get_state(followup_config).values["messages"]
print("저장된 메시지", len(saved))
print("첫 자기소개 보존:", saved[0].content == QUESTIONS[0])

Document Loader는 책 한 권을 읽어서 내용을 컴퓨터가 이해할 수 있는 글자로 바꾸는 역할이고,  
Text Splitter는 그 책 내용을 한 번에 다 읽기 어려우니까 여러 페이지나 챕터로 나누는 역할이야.  
즉, 먼저 문서를 불러오고(Document Loader), 그 다음에 나누는(Text Splitter) 과정이라고 생각하면 돼!
저장된 메시지 8
첫 자기소개 보존: True


## 3. 프로필 삭제와 대화 원문 구분하기

### 3-1. 프로필·원문을 따로 준비

- **프로필 `profile`**: `name` 같은 항목별 정보를 담은 딕셔너리.
- **대화 원문 `history`**: 사용자·AI 메시지 객체를 담은 리스트. 이름이 문장 속에도 포함됨.
- **독립 자료**: 두 자료는 별도로 생성. 이 보충 예제는 앞 Agent의 체크포인터나 메시지를 수정하지 않음.


In [43]:
profile = {"name": "영희"}
history = [HumanMessage(content="내 이름은 영희야"), AIMessage(content="영희님, 반갑습니다.")]
print(profile)

{'name': '영희'}


### 3-2. 프로필 삭제 후 대화 원문 확인

- **`dict.pop(key, default)`**: 딕셔너리에서 키를 제거하고, 제거한 값을 반환.
- **현재 결과**: `profile`에서 `name` 제거, 반환된 ‘민수’는 `removed_name`에 저장.
- **기본값 `None`**: 키가 이미 없으면 오류 대신 `None` 반환.
- **원문 보존**: 별도 리스트 `history`에는 메시지 객체 두 개와 문장 속 이름이 그대로 남음.
- **삭제 범위**: 프로필·대화 원문·체크포인트 등 정보가 보관된 위치별로 삭제 대상을 구분.


In [44]:
removed_name = profile.pop("name")
print("꺼낸 값:", removed_name)
print("삭제 후 프로필:", profile)

print("원본 메시지 수:", len(history))
for message in history:
    print(message.type, message.content)

꺼낸 값: 영희
삭제 후 프로필: {}
원본 메시지 수: 2
human 내 이름은 영희야
ai 영희님, 반갑습니다.
